In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import datetime

from memory.short_term_memory import ShortTermMemory
from memory.long_term_memory import LongTermMemory

from tools.facts_retrieve import retrieve_facts_schema
from function_call import select_service
from prompts import SYSTEM_PROMPT

load_dotenv()
client = OpenAI(
    base_url="https://freellmapi-seyc.onrender.com/v1",
    api_key=os.environ.get("FREE_LLM_API")
)


stm = ShortTermMemory(mode ="summary_buffer", llm_client=client)
ltm = LongTermMemory(llm_client=client)


session_id = ltm.session_id

In [5]:
user_input = input("Input your query: ")
stm.add(assistant_msg={"role": "user", "content": user_input}, role="tool")
messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat
while True:
    if user_input.strip().lower() == "exit":
        ltm.add(stm.slide_chat)
        break

    print("\n\n"+str(messages[1:]))
    response = client.chat.completions.create(
        model="auto",
        messages=messages,
        tools=[retrieve_facts_schema],
        tool_choice='auto'
    )
    msg = response.choices[0].message

    print(response.choices[0].message)


    if msg.tool_calls:
        assistant_tool_msg = {
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        }
        stm.add(assistant_msg=assistant_tool_msg, role="tool")

        for tool_call in msg.tool_calls:
            tool_result = select_service(tool_call.function)
            stm.add(
                assistant_msg={
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                },
                role="tool",
            )
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat

    if response.choices[0].finish_reason == "stop":
        stm.add(assistant_msg={"role": "assistant", "content": msg.content}, role="tool")
        user_input = input("Input your query: ")
        stm.add(assistant_msg={"role": "user", "content": user_input}, role="tool")
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat



[{'role': 'user', 'content': 'Hi'}]
ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to respond as a helpful AI. The user just said "Hi". It\'s greeting. No need to retrieve memory. Just greet back. Possibly ask how can help.')


[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}, {'role': 'user', 'content': 'Who are you?'}]
ChatCompletionMessage(content='I’m ChatGPT, an AI language model created by OpenAI. I’m here to help answer your questions, brainstorm ideas, explain concepts, draft text, troubleshoot problems, and more—just let me know what you need!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User asks "Who are you?" They want description of assistant. Should answer briefly. No need to retrieve memory.')


[{'role': 'use

In [6]:
stm.slide_chat

[{'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Who are you?'},
 {'role': 'assistant',
  'content': 'I’m ChatGPT, an AI language model created by OpenAI. I’m here to help answer your questions, brainstorm ideas, explain concepts, draft text, troubleshoot problems, and more—just let me know what you need!'},
 {'role': 'user', 'content': 'WHat is my name?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'fc_f63a4824-bfdf-40d6-bc36-014310559df4',
    'function': {'arguments': '{"query":"user name","top_k":5}',
     'name': 'retrieve_facts'},
    'type': 'function'}]},
 {'role': 'tool',
  'tool_call_id': 'fc_f63a4824-bfdf-40d6-bc36-014310559df4',
  'content': '["The user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "User\'s name is Satyam Sharma"]'},
 {'role': 'assistant', 'content': 'Your name is Sat

In [7]:
messages

[{'role': 'system',
  'content': 'You are a helpful AI assistant with access to memory about this user, built from past conversations.\n\nYou have one memory tools:\n\n- **retrieve_facts**: retrieves distilled facts about the user (preferences, identity, constraints — e.g. "user prefers concise answers", "user works with Python"). Use this when the user\'s current request could be informed by something you may already know about them, their preferences, or their working context.\n\n## When to search memory\n\nSearch BEFORE answering, not after, when:\n- The user asks about their own preferences, past decisions, or history with you\n- The user references something implicitly ("the usual approach", "like before")\n- Answering well requires knowing something specific about this user that a generic answer wouldn\'t capture\n\nDo NOT search memory for:\n- Generic factual/technical questions unrelated to the user\'s personal context\n- Simple greetings or small talk\n\n## After retrieving\n\